In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tombackert/brain-tumor-mri-data")

print("Path to dataset files:", path)

In [ ]:
import tensorflow
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks

data_set = '!!!!Train Folder!!!'
batch = 32

training_data_set = image_dataset_from_directory(
    data_set,
    batch_size = 32,
    image_size = (224, 224),
    subset = 'training',
    label_mode = 'categorical',
    validation_split = 0.2,
    shuffle = True,
    seed = 123
)

validation_data_set = image_dataset_from_directory(
    data_set,
    batch_size = 32,
    image_size = (224, 224),
    subset = 'validation',
    label_mode = 'categorical',
    validation_split = 0.2,
    shuffle = True,
    seed = 123
)


mobile_model = EfficientNetB0(
    weights = 'imagenet',
    input_shape = (224, 224, 3),
    include_top = False
)

mobile_model.trainable = False



main_model = models.Sequential([
    mobile_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(4, activation='softmax')
])

main_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

reduce_speed = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-6
)

stop_it = callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 4,
    restore_best_weights= True
)


history = main_model.fit(
    training_data_set,
    validation_data = validation_data_set,
    epochs = 10,
    callbacks = [reduce_speed, stop_it]
)

main_model.save('brain-tumor-model.h5')